In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q kaggle

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "evangregor"
os.environ['KAGGLE_KEY'] = "KGAT_ee7af7d4f30a6f968e534c5b38566789"

!kaggle datasets download -d abdohamdg/fracatlas-dataset
!unzip -q fracatlas-dataset.zip -d /content/fracatlas_raw

wrong = "/content/fracatlas_raw/Fracatlas/validation/lables"
right = "/content/fracatlas_raw/Fracatlas/validation/labels"

if os.path.exists(wrong):
    os.rename(wrong, right)
    print("✅ Fixed 'lables' → 'labels'")
else:
    print("✔️ No typo found")

Dataset URL: https://www.kaggle.com/datasets/abdohamdg/fracatlas-dataset
License(s): ODC Public Domain Dedication and Licence (PDDL)
100% 87.6M/87.6M [00:00<00:00, 111MB/s]

✅ Fixed 'lables' → 'labels'


In [ ]:
!pip install timm --quiet

In [ ]:
FRACATLAS = Path("/content/fracatlas_raw/Fracatlas")
yaml_content = f"""path  : {FRACATLAS}
train : train/images
val   : validation/images
test  : test/images

nc    : 1
names : ["fractured"]
"""

yaml_path = FRACATLAS / "data.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print("✅ data.yaml fixed")

✅ data.yaml fixed


In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
import timm
import cv2

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/mura_dataset.zip"
extract_path = "/content/mura_dataset"

# create folder
os.makedirs(extract_path, exist_ok=True)

# unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted to:", extract_path)

✅ Dataset extracted to: /content/mura_dataset


In [ ]:
MURA_ROOT      = Path("/content/mura_dataset/MURA-v1.1")
MURA_TRAIN_CSV = MURA_ROOT / "train_labeled_studies.csv"
MURA_VAL_CSV   = MURA_ROOT / "valid_labeled_studies.csv"
FRACATLAS_ROOT = Path("/content/fracatlas_raw/Fracatlas")
SAVE_PATH      = Path("/content/drive/MyDrive/models/best_effnet_v4.pth")

In [ ]:
CONFIG = {
    "model_name"      : "efficientnet_b2",
    "num_classes"     : 2,
    "img_size"        : 260,
    "batch_size"      : 32,          # increase to 64 on P100
    "num_epochs"      : 40,
    "lr"              : 3e-5,
    "weight_decay"    : 1e-4,
    "patience"        : 10,
    "num_workers"     : 2,
    "drop_rate"       : 0.5,
    "class_weights"   : [1.0, 2.5],  # slight abnormal boost
    "label_smoothing" : 0.1,
    "mixup_alpha"     : 0.3,
    "warmup_epochs"   : 3,
    "infer_threshold" : 0.6623,       # Youden optimal from previous run
    "use_clahe"       : True,
    "border_crop_pct" : 0.04,
    "save_path"       : str(SAVE_PATH),
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"Save to : {CONFIG['save_path']}")

Device  : cpu
Save to : /content/drive/MyDrive/models/best_effnet_v4.pth


In [ ]:
def remove_xray_border(pil_img, pct=CONFIG["border_crop_pct"]):
    img    = np.array(pil_img)
    h, w   = img.shape[:2]
    mh, mw = int(h * pct), int(w * pct)
    return Image.fromarray(img[mh:h-mh, mw:w-mw])

def apply_clahe(pil_img):
    img_np   = np.array(pil_img.convert("L"))
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img_np)
    result   = Image.fromarray(enhanced)
    return remove_xray_border(result)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

infer_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class MuraDataset(Dataset):
    """
    MURA: labels come from folder name (positive/negative).
    Returns (image_tensor, label, study_id) for study-level eval.
    """
    def __init__(self, csv_file, root_dir, transform=None, use_clahe=True):
        self.data      = pd.read_csv(csv_file, header=None)
        self.root_dir  = Path(root_dir)
        self.transform = transform
        self.use_clahe = use_clahe
        self.samples   = []

        for path in self.data[0]:
            label     = 1 if "positive" in path else 0
            full_path = self.root_dir / path
            if not full_path.is_dir():
                continue
            study_id = path
            for img_name in sorted(os.listdir(full_path)):
                if img_name.endswith(".png") and not img_name.startswith("._"):
                    self.samples.append(
                        (str(full_path / img_name), label, study_id)
                    )

        n0 = sum(1 for _,l,_ in self.samples if l==0)
        n1 = sum(1 for _,l,_ in self.samples if l==1)
        print(f"  MURA    : {len(self.samples):6d} images | Normal: {n0} | Abnormal: {n1}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, study_id = self.samples[idx]
        image = Image.open(img_path).convert("L")
        if self.use_clahe:
            image = apply_clahe(image)
        if self.transform:
            image = self.transform(image)
        return image, label, study_id

In [ ]:
class FracAtlasDataset(Dataset):
    """
    FracAtlas: label derived from YOLO label file.
    Non-empty label file = fracture (1), empty = normal (0).
    Returns (image_tensor, label, study_id) — study_id = filename.
    """
    def __init__(self, root_dir, split="train",
                 transform=None, use_clahe=True):
        self.root      = Path(root_dir)
        self.transform = transform
        self.use_clahe = use_clahe
        self.samples   = []

        img_dir = self.root / split / "images"
        lbl_dir = self.root / split / "labels"

        if not img_dir.exists():
            print(f"  ⚠ FracAtlas {split} not found at {img_dir}")
            return

        for img_path in sorted(img_dir.glob("*.jpg")):
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            # label: 1 if label file has content, 0 if empty/missing
            if lbl_path.exists():
                label = 1 if lbl_path.stat().st_size > 0 else 0
            else:
                label = 0
            self.samples.append((str(img_path), label, img_path.stem))

        n0 = sum(1 for _,l,_ in self.samples if l==0)
        n1 = sum(1 for _,l,_ in self.samples if l==1)
        print(f"  FracAtlas [{split}]: {len(self.samples):5d} images | "
              f"Normal: {n0} | Fractured: {n1}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, study_id = self.samples[idx]
        image = Image.open(img_path).convert("L")
        if self.use_clahe:
            image = apply_clahe(image)
        if self.transform:
            image = self.transform(image)
        return image, label, study_id

In [ ]:
class CombinedDataset(Dataset):
    def __init__(self, datasets):
        self.datasets = datasets
        self.lengths  = [len(d) for d in datasets]
        self.offsets  = [0] + list(np.cumsum(self.lengths[:-1]))

    def __len__(self):
        return sum(self.lengths)

    def __getitem__(self, idx):
        for i, (offset, length) in enumerate(
            zip(self.offsets, self.lengths)
        ):
            if idx < offset + length:
                return self.datasets[i][idx - offset]
        raise IndexError(f"Index {idx} out of range")

In [ ]:
print("\nLoading datasets...")

mura_train = MuraDataset(MURA_TRAIN_CSV, MURA_ROOT.parent,
                          transform=train_transform,
                          use_clahe=CONFIG["use_clahe"])
mura_val   = MuraDataset(MURA_VAL_CSV,   MURA_ROOT.parent,
                          transform=val_transform,
                          use_clahe=CONFIG["use_clahe"])

frac_train = FracAtlasDataset(FRACATLAS_ROOT, split="train",
                               transform=train_transform,
                               use_clahe=CONFIG["use_clahe"])
frac_val   = FracAtlasDataset(FRACATLAS_ROOT, split="validation",
                               transform=val_transform,
                               use_clahe=CONFIG["use_clahe"])
frac_test  = FracAtlasDataset(FRACATLAS_ROOT, split="test",
                               transform=val_transform,
                               use_clahe=CONFIG["use_clahe"])

# Combine
train_dataset = CombinedDataset([mura_train, frac_train])
val_dataset   = CombinedDataset([mura_val,   frac_val])

print(f"\n  Combined train : {len(train_dataset):6d} images")
print(f"  Combined val   : {len(val_dataset):6d} images")
print(f"  FracAtlas test : {len(frac_test):6d} images (held out)")

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"],
                          pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)
test_loader  = DataLoader(frac_test,     batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True)


Loading datasets...
  MURA    :  36808 images | Normal: 21935 | Abnormal: 14873
  MURA    :   3197 images | Normal: 1667 | Abnormal: 1530
  FracAtlas [train]:   574 images | Normal: 0 | Fractured: 574
  FracAtlas [validation]:    82 images | Normal: 0 | Fractured: 82
  FracAtlas [test]:    61 images | Normal: 0 | Fractured: 61

  Combined train :  37382 images
  Combined val   :   3279 images
  FracAtlas test :     61 images (held out)


In [ ]:
model = timm.create_model(
    CONFIG["model_name"],
    pretrained  = True,
    num_classes = CONFIG["num_classes"],
    drop_rate   = CONFIG["drop_rate"],
)
model = model.to(device)
print(f"\nModel: {CONFIG['model_name']} | "
      f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]


Model: efficientnet_b2 | Params: 7,703,812


In [ ]:
ckpt_path = CONFIG["save_path"]  # e.g. /content/drive/MyDrive/models/best_effnet_v4.pth

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

# Use config stored inside checkpoint (VERY IMPORTANT)
cfg = ckpt["config"]

model = timm.create_model(
    cfg["model_name"],
    pretrained=False,              # ❗ don't load imagenet again
    num_classes=cfg["num_classes"],
    drop_rate=cfg["drop_rate"],
)

model.load_state_dict(ckpt["model_state"])
model = model.to(device)
model.eval()

print(f"\n✅ Loaded model from epoch {ckpt['epoch']}")
print(f"AUC: {ckpt['val_auc']:.4f} | Acc: {ckpt['val_acc']:.4f}")


✅ Loaded model from epoch 19
AUC: 0.9134 | Acc: 0.8517


In [ ]:
class_weights = torch.tensor(CONFIG["class_weights"],
                              dtype=torch.float).to(device)
criterion     = nn.CrossEntropyLoss(
    weight         = class_weights,
    label_smoothing= CONFIG["label_smoothing"]
)
optimizer     = torch.optim.AdamW(
    model.parameters(),
    lr           = CONFIG["lr"],
    weight_decay = CONFIG["weight_decay"]
)
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0,
    total_iters=CONFIG["warmup_epochs"]
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["num_epochs"] - CONFIG["warmup_epochs"],
    eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine],
    milestones=[CONFIG["warmup_epochs"]]
)

In [ ]:
def mixup_data(x, y, alpha=CONFIG["mixup_alpha"]):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(x.size(0)).to(x.device)
    mix_x = lam * x + (1 - lam) * x[idx]
    return mix_x, y, y[idx], lam

def mixup_criterion(pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
def aggregate_study_level(all_probs, all_labels, all_study_ids, threshold):
    study_probs  = defaultdict(list)
    study_labels = {}
    for prob, label, sid in zip(all_probs, all_labels, all_study_ids):
        study_probs[sid].append(float(prob))
        study_labels[sid] = int(label)

    final_probs  = [max(study_probs[s])  for s in study_probs]
    final_labels = [study_labels[s]      for s in study_probs]
    final_preds  = [1 if p >= threshold else 0 for p in final_probs]
    return final_probs, final_labels, final_preds

In [ ]:
history        = {"train_loss":[], "val_loss":[], "val_auc":[], "val_acc":[]}
best_val_auc   = 0.0
patience_count = 0

print("\n" + "="*60)
print("Training Start — MURA + FracAtlas Combined")
print("="*60)

for epoch in range(1, CONFIG["num_epochs"] + 1):

    # ── TRAIN ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for images, labels, _ in tqdm(train_loader,
                                   desc=f"Ep {epoch:02d} [Train]",
                                   leave=False):
        images, labels = images.to(device), labels.to(device)
        mix_images, y_a, y_b, lam = mixup_data(images, labels)

        optimizer.zero_grad()
        outputs = model(mix_images)
        loss    = mixup_criterion(outputs, y_a, y_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # ── VALIDATE ─────────────────────────────────────────────
    model.eval()
    val_loss     = 0.0
    all_probs    = []
    all_labels   = []
    all_study_ids= []

    with torch.no_grad():
        for images, labels, study_ids in tqdm(val_loader,
                                               desc=f"Ep {epoch:02d} [Val]  ",
                                               leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            for prob, label, sid in zip(probs, labels.cpu().numpy(), study_ids):
                all_probs.append(float(prob))
                all_labels.append(int(label))
                all_study_ids.append(sid)

    val_loss /= len(val_loader)

    # Study-level aggregation
    final_probs, final_labels, final_preds = aggregate_study_level(
        all_probs, all_labels, all_study_ids, CONFIG["infer_threshold"]
    )

    val_acc = accuracy_score(final_labels, final_preds)
    val_auc = roc_auc_score(final_labels, final_probs)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)
    history["val_acc"].append(val_acc)

    print(f"Ep {epoch:02d} | TrLoss {train_loss:.4f} | "
          f"ValLoss {val_loss:.4f} | Acc {val_acc*100:.1f}% | "
          f"AUC {val_auc:.4f} | LR {current_lr:.1e}")

    # ── CHECKPOINT ───────────────────────────────────────────
    if val_auc > best_val_auc:
        best_val_auc   = val_auc
        patience_count = 0
        os.makedirs(Path(CONFIG["save_path"]).parent, exist_ok=True)
        torch.save({
            "epoch"      : epoch,
            "model_state": model.state_dict(),
            "optimizer"  : optimizer.state_dict(),
            "val_auc"    : val_auc,
            "val_acc"    : val_acc,
            "config"     : CONFIG,
        }, CONFIG["save_path"])
        print(f"  ✅ Saved (AUC {val_auc:.4f})")
    else:
        patience_count += 1
        if patience_count >= CONFIG["patience"]:
            print(f"\n⛔ Early stop at epoch {epoch}. Best AUC: {best_val_auc:.4f}")
            break


Training Start — MURA + FracAtlas Combined


KeyboardInterrupt: 

In [ ]:
def evaluate(loader, name):
    all_probs     = []
    all_labels    = []
    all_study_ids = []

    with torch.no_grad():
        for images, labels, study_ids in tqdm(loader, desc=name):
            images  = images.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()

            for prob, label, sid in zip(probs, labels.numpy(), study_ids):
                all_probs.append(float(prob))
                all_labels.append(int(label))
                all_study_ids.append(sid)

    fp, fl, fpred = aggregate_study_level(
        all_probs, all_labels, all_study_ids, CONFIG["infer_threshold"]
    )

    print(f"\n── {name} ──")

    # ---------- AUC ----------
    try:
        auc = roc_auc_score(fl, fp)
        print(f"  AUC-ROC     : {auc:.4f}")
    except:
        auc = None
        print("  AUC-ROC     : ❌ Undefined (single class)")

    # ---------- Accuracy ----------
    acc = accuracy_score(fl, fpred)
    print(f"  Accuracy    : {acc*100:.2f}%")

    # ---------- Confusion Matrix ----------
    cm = confusion_matrix(fl, fpred)

    if cm.shape == (2, 2):
        tn, fp_n, fn, tp = cm.ravel()

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp_n) if (tn + fp_n) > 0 else 0

        print(f"  Sensitivity : {sens*100:.2f}%")
        print(f"  Specificity : {spec*100:.2f}%")

    else:
        # Only one class present
        print("  ⚠️ Only one class present in dataset")

        if len(set(fl)) == 1 and fl[0] == 1:
            # Only positives → compute sensitivity
            tp = sum((np.array(fl) == 1) & (np.array(fpred) == 1))
            fn = sum((np.array(fl) == 1) & (np.array(fpred) == 0))
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0

            print(f"  Sensitivity : {sens*100:.2f}%")
            print("  Specificity : ❌ Undefined")

            spec = None

        else:
            sens = None
            spec = None

    # ---------- Classification Report ----------
    try:
        print(classification_report(fl, fpred,
                                   target_names=["Normal","Abnormal"]))
    except:
        print("  ⚠️ Classification report not available")

    return auc, sens, spec

In [ ]:
auc_val, sens_val, spec_val = evaluate(val_loader, "Combined Val (MURA + FracAtlas)")

Combined Val (MURA + FracAtlas): 100%|██████████| 103/103 [10:06<00:00,  5.89s/it]


── Combined Val (MURA + FracAtlas) ──
  AUC-ROC     : 0.6788
  Accuracy    : 61.75%
  Sensitivity : 64.84%
  Specificity : 58.85%
              precision    recall  f1-score   support

      Normal       0.64      0.59      0.61       661
    Abnormal       0.60      0.65      0.62       620

    accuracy                           0.62      1281
   macro avg       0.62      0.62      0.62      1281
weighted avg       0.62      0.62      0.62      1281



In [ ]:
auc_test, sens_test, spec_test = evaluate(test_loader, "FracAtlas Test (multi-body only)")

FracAtlas Test (multi-body only): 100%|██████████| 2/2 [00:14<00:00,  7.16s/it]


── FracAtlas Test (multi-body only) ──
  AUC-ROC     : nan
  Accuracy    : 31.15%
  Sensitivity : 31.15%
  Specificity : 0.00%
              precision    recall  f1-score   support

      Normal       0.00      0.00      0.00         0
    Abnormal       1.00      0.31      0.47        61

    accuracy                           0.31        61
   macro avg       0.50      0.16      0.24        61
weighted avg       1.00      0.31      0.47        61



In [ ]:
# Collect data
all_probs, all_labels, all_ids = [], [], []

with torch.no_grad():
    for images, labels, study_ids in val_loader:
        images = images.to(device)
        probs = torch.softmax(model(images), dim=1)[:, 1]

        for p, l, sid in zip(probs.cpu().numpy(), labels.numpy(), study_ids):
            all_probs.append(p)
            all_labels.append(l)
            all_ids.append(sid)

# Aggregate
fp, fl, _ = aggregate_study_level(all_probs, all_labels, all_ids, 0.5)

# ROC
fpr, tpr, _ = roc_curve(fl, fp)
auc_score = roc_auc_score(fl, fp)

In [ ]:
print("\n" + "="*60)
print("Model Comparison: v3 (MURA only) vs v4 (MURA + FracAtlas)")
print("="*60)
print(f"{'Metric':<25} {'v3 (MURA only)':>16} {'v4 (combined)':>16}")
print("-" * 60)
print(f"{'AUC-ROC':<25} {'0.8933':>16} {auc_val:>16.4f}")
print(f"{'Sensitivity':<25} {'80.1%':>16} {sens_val*100:>15.1f}%")
print(f"{'Specificity':<25} {'87.6%':>16} {spec_val*100:>15.1f}%")
print(f"{'FracAtlas Test AUC':<25} {'N/A (not tested)':>16} {auc_test:>16.4f}")
print("="*60)
print("\n✅ Done. Update ensemble config:")
print(f'  "effnet_ckpt" : "{CONFIG["save_path"]}",')



Model Comparison: v3 (MURA only) vs v4 (MURA + FracAtlas)
Metric                      v3 (MURA only)    v4 (combined)
------------------------------------------------------------
AUC-ROC                             0.8933           0.6788
Sensitivity                          80.1%            64.8%
Specificity                          87.6%            58.9%
FracAtlas Test AUC        N/A (not tested)              nan

✅ Done. Update ensemble config:
  "effnet_ckpt" : "/content/drive/MyDrive/models/best_effnet_v4.pth",


In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

# Use STUDY-level predictions (important)
fpr, tpr, thresholds = roc_curve(fl, fp)

specificity = 1 - fpr
youden_j = tpr + specificity - 1

best_idx = np.argmax(youden_j)

best_thresh = thresholds[best_idx]
best_sens   = tpr[best_idx]
best_spec   = specificity[best_idx]

print("\n🔥 Best Threshold (Youden’s J):")
print(f"Threshold    : {best_thresh:.4f}")
print(f"Sensitivity  : {best_sens*100:.2f}%")
print(f"Specificity  : {best_spec*100:.2f}%")


🔥 Best Threshold (Youden’s J):
Threshold    : 0.8011
Sensitivity  : 51.77%
Specificity  : 77.76%


In [ ]:
# Distance to perfect classifier (0,1)
dist = np.sqrt((fpr)**2 + (1 - tpr)**2)

best_idx = np.argmin(dist)

best_thresh = thresholds[best_idx]
best_sens   = tpr[best_idx]
best_spec   = 1 - fpr[best_idx]

print("\n🔥 Best Threshold (Closest to Ideal):")
print(f"Threshold    : {best_thresh:.4f}")
print(f"Sensitivity  : {best_sens*100:.2f}%")
print(f"Specificity  : {best_spec*100:.2f}%")


🔥 Best Threshold (Closest to Ideal):
Threshold    : 0.7172
Sensitivity  : 60.00%
Specificity  : 67.93%


In [ ]:
alpha = 0.6  # prioritize sensitivity (0.5 = equal)

score = alpha * tpr + (1 - alpha) * (1 - fpr)

best_idx = np.argmax(score)

best_thresh = thresholds[best_idx]
best_sens   = tpr[best_idx]
best_spec   = 1 - fpr[best_idx]

print("\n🔥 Best Threshold (Weighted):")
print(f"Threshold    : {best_thresh:.4f}")
print(f"Sensitivity  : {best_sens*100:.2f}%")
print(f"Specificity  : {best_spec*100:.2f}%")


🔥 Best Threshold (Weighted):
Threshold    : 0.5057
Sensitivity  : 80.16%
Specificity  : 39.49%


In [ ]:
!pip install grad-cam -q

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import cv2

In [ ]:
target_layer = model.conv_head

In [ ]:
images, labels, _ = next(iter(val_loader))
image = images[0].unsqueeze(0).to(device)
label = labels[0].item()

In [ ]:
cam = GradCAM(model=model, target_layers=[target_layer])

grayscale_cam = cam(input_tensor=image)[0]

In [ ]:
# Convert tensor → numpy image
img = images[0].permute(1, 2, 0).cpu().numpy()

# Normalize to 0–1
img = (img - img.min()) / (img.max() - img.min())

# Overlay heatmap
visualization = show_cam_on_image(img, grayscale_cam, use_rgb=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.title("Original")
plt.imshow(img)
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Grad-CAM")
plt.imshow(visualization)
plt.axis("off")

plt.show()